# Modelado para detección de fraccionamiento

## Dependencias

In [1]:
import os

import polars as pl
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import entropy
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score 

import mlflow
import mlflow.sklearn
import mlflow.pyfunc

## Funciones auxiliares

In [2]:
def calculate_amount_entropy(amounts) -> float:
    # Convertir Series de Polars a lista de Python si es necesario
    if hasattr(amounts, 'to_list'):  # Es un Series de Polars
        amounts_list = amounts.to_list()
    else:  # Ya es una lista de Python
        amounts_list = amounts
    
    if not amounts_list:  # Ahora podemos usar if not con una lista
        return 0.0
    
    value_counts = pd.Series(amounts_list).value_counts(normalize=True)
    return entropy(value_counts)

In [3]:
calculate_amount_entropy_pl = pl.col("transaction_amount").map_batches(
    lambda s: [calculate_amount_entropy(val) for val in s]
).alias("amount_entropy")

In [4]:
BASE_DIR = "../"
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
FILE_NAME = os.listdir(RAW_DATA_DIR)[0]
FILE_PATH = os.path.join(RAW_DATA_DIR, FILE_NAME)

## Carga datos

In [6]:
print("Cargando y preprocesando datos...")
try:
    df_transactions = pl.read_parquet(FILE_PATH)
    df_transactions = df_transactions.with_columns([
        pl.col("transaction_amount").cast(pl.Float64),
        pl.col("transaction_date").cast(pl.Datetime)
    ]).unique() # Eliminar duplicados
    print(f"Datos cargados. Dimensiones: {df_transactions.shape}")
except Exception as e:
    print(f"Error cargando o preprocesando el archivo: {e}")

Cargando y preprocesando datos...
Datos cargados. Dimensiones: (10758414, 8)


In [ ]:
# Extraer componentes de fecha para el análisis temporal 
df_transactions = df_transactions.with_columns([
    pl.col("transaction_date").dt.hour().alias("hour"),
    pl.col("transaction_date").dt.weekday().alias("day_of_week"),
    pl.col("transaction_date").dt.month().alias("month"),
    pl.col("transaction_date").dt.date().alias("transaction_day")
])

## Filtrado por heurísticas y Feature Engineering

In [ ]:
print("Generando grupos candidatos y features avanzadas...")

# Definir la ventana de tiempo para el fraccionamiento
WINDOW_DURATION = pl.duration(hours=24)
MIN_TRANSACTIONS_IN_GROUP = 3 # Un umbral inicial de al menos 3 transacciones

candidate_groups_df = df_transactions.group_by_dynamic(
    index_column="transaction_date",
    every=WINDOW_DURATION,
    by=["user_id", "merchant_id", "transaction_type"] # Claves para identificar el grupo
).agg([
    pl.count().alias("transaction_count"),
    pl.col("transaction_amount").sum().alias("total_group_amount"),
    pl.col("transaction_amount").std().alias("amount_stdev"),
    pl.col("transaction_amount").median().alias("amount_median"),
    pl.col("transaction_amount").min().alias("amount_min"),
    pl.col("transaction_amount").max().alias("amount_max"),
    (pl.col("transaction_date").max() - pl.col("transaction_date").min()).dt.total_minutes().alias("time_span_minutes"),
    pl.col("transaction_amount").list().alias("all_amounts_in_group"), # Para entropía/CV
    pl.col("transaction_date").list().alias("all_dates_in_group") # Para avg_time_delta
]).filter(
    pl.col("transaction_count") >= MIN_TRANSACTIONS_IN_GROUP
).with_columns([
    # Coeficiente de Variación: std / mean. Cuidado con mean=0
    (pl.col("amount_stdev") / pl.col("total_group_amount") * pl.col("transaction_count")).alias("amount_coeff_of_variation").fill_nan(0.0), # Rellenar NaN si std/mean es 0/0
    # Entropía de los Montos
    pl.col("all_amounts_in_group").apply(calculate_amount_entropy).alias("amount_entropy"),
    # Calcular promedio de delta de tiempo entre transacciones dentro del grupo
    pl.col("all_dates_in_group").apply(
        lambda dates: np.mean(np.diff(np.sort([d.timestamp() for d in dates]))) / 60 if len(dates) > 1 else 0.0
    ).alias("avg_time_delta_minutes"),
    # Flag para montos redondos (ej. múltiplos de 100000)
    pl.col("all_amounts_in_group").apply(
        lambda amounts: any(a % 100000 == 0 for a in amounts)
    ).alias("has_round_amounts")
])
# Limpiar columnas auxiliares
candidate_groups_df = candidate_groups_df.drop(["all_amounts_in_group", "all_dates_in_group"])

print(f"Total de grupos candidatos para ML: {candidate_groups_df.shape[0]}")
print(candidate_groups_df.head())


## Normalización y Encoding

In [ ]:
print("Preparando features para los modelos ML...")

# Definir las features numéricas que serán usadas por IF y DBSCAN
# Excluir IDs o features ya agregadas que no son numéricas para el modelo
numerical_features = [
    "transaction_count",
    "total_group_amount",
    "amount_stdev",
    "amount_median",
    "amount_min",
    "amount_max",
    "time_span_minutes",
    "amount_coeff_of_variation",
    "amount_entropy",
    "avg_time_delta_minutes"
]

# Asegurarse de que todas las columnas existan antes de seleccionar
numerical_features_exist = [col for col in numerical_features if col in candidate_groups_df.columns]
if len(numerical_features_exist) != len(numerical_features):
    print(f"Advertencia: Algunas features numéricas esperadas no existen. Usando: {numerical_features_exist}")
numerical_features = numerical_features_exist

X_candidates_polars = candidate_groups_df.select(numerical_features)

# Convertir a NumPy para Scikit-learn
X_candidates = X_candidates_polars.to_numpy()

# Normalización con StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_candidates)
print(f"Features escaladas. Dimensiones de X_scaled: {X_scaled.shape}")

## Modelos

In [ ]:

# Configuración de MLFlow
# Si usas un servidor MLFlow remoto, configura mlflow.set_tracking_uri()
mlflow.set_experiment("Deteccion_Fraccionamiento_Nequi")

# Parámetros para iterar (ejemplo, puedes añadir más o usar un grid search más formal)
if_params_list = [
    {"n_estimators": 100, "contamination": 'auto', "random_state": 42},
    {"n_estimators": 200, "contamination": 0.01, "random_state": 42},
]

dbscan_params_list = [
    {"eps": 0.5, "min_samples": 5},
    {"eps": 0.3, "min_samples": 10},
]

# Iterar sobre las combinaciones de parámetros para encontrar el mejor modelo
run_id_counter = 0
for if_params in if_params_list:
    for dbscan_params in dbscan_params_list:
        run_id_counter += 1
        with mlflow.start_run(run_name=f"Run_{run_id_counter}"):
            print(f"\n--- Ejecutando MLFlow Run {run_id_counter} ---")
            print(f"IF Params: {if_params}")
            print(f"DBSCAN Params: {dbscan_params}")

            # Logear parámetros en MLFlow
            mlflow.log_params({"if_" + k: v for k, v in if_params.items()})
            mlflow.log_params({"dbscan_" + k: v for k, v in dbscan_params.items()})

            # --- Entrenamiento de Isolation Forest ---
            iso_forest = IsolationForest(**if_params)
            iso_forest.fit(X_scaled)
            # anomaly_scores_if: Menor valor = Más anómalo
            anomaly_scores_if = iso_forest.decision_function(X_scaled)
            # Convertir a un score de riesgo: Mayor valor = Más riesgo (normalizar 0-1)
            min_score, max_score = anomaly_scores_if.min(), anomaly_scores_if.max()
            risk_scores_if = 1 - (anomaly_scores_if - min_score) / (max_score - min_score)
            
            # --- Entrenamiento de DBSCAN ---
            dbscan = DBSCAN(**dbscan_params)
            # labels: -1 para ruido (anomalías), 0, 1, ... para clústeres
            dbscan_labels = dbscan.fit_predict(X_scaled)
            # is_noise: 1 si es ruido, 0 si es parte de un clúster
            is_dbscan_noise = (dbscan_labels == -1).astype(int)

            # --- Ensamble de Scores ---
            # Score combinado: Puedes ponderar o usar una regla lógica (ej. AND)
            # Aquí usaremos una combinación simple de los scores normalizados
            # Ponderación 0.7 para IF, 0.3 para DBSCAN (ajustable)
            combined_score = 0.7 * risk_scores_if + 0.3 * is_dbscan_noise

            # Añadir scores al DataFrame de candidatos (para análisis posterior)
            candidate_groups_with_scores = candidate_groups_df.with_columns([
                pl.Series("isolation_forest_score", anomaly_scores_if),
                pl.Series("dbscan_label", dbscan_labels),
                pl.Series("is_dbscan_noise", is_dbscan_noise),
                pl.Series("combined_risk_score", combined_score)
            ])

            # --- Evaluación y Registro de Métricas ---
            # Para modelos no supervisados sin ground truth, la evaluación es desafiante.
            # Aquí usaremos métricas intrínsecas y un umbral para simular Precision/Recall.

            # Métrica intrínseca de clustering (para DBSCAN)
            # silhouette_score requiere al menos 2 clústeres y 2 muestras.
            try:
                if len(np.unique(dbscan_labels)) > 1 and len(dbscan_labels) > 1:
                    silhouette_avg = silhouette_score(X_scaled, dbscan_labels)
                    mlflow.log_metric("dbscan_silhouette_score", silhouette_avg)
                    print(f"DBSCAN Silhouette Score: {silhouette_avg:.4f}")
                else:
                    print("No suficientes clústeres para calcular Silhouette Score.")
            except Exception as e:
                print(f"Error calculando Silhouette Score: {e}")

            # Umbral de riesgo combinado (para simular alertas)
            # Por ejemplo, top 5% de los scores más altos son alertas
            alert_threshold = np.percentile(combined_score, 95) # Top 5% más riesgoso
            num_alerts = (combined_score >= alert_threshold).sum()
            
            mlflow.log_metric("alert_threshold", alert_threshold)
            mlflow.log_metric("num_alerts", num_alerts)
            print(f"Alertas generadas (top 5%): {num_alerts}")

            # Evaluación de la "Calidad" de las Alertas sin etiquetas reales:
            # Puedes examinar las características de los grupos de "alto riesgo"
            high_risk_groups = candidate_groups_with_scores.filter(
                pl.col("combined_risk_score") >= alert_threshold
            )
            
            # Puedes calcular la "uniformidad promedio" de los montos en los grupos de alto riesgo
            avg_entropy_high_risk = high_risk_groups.select("amount_entropy").mean().item()
            avg_coeff_var_high_risk = high_risk_groups.select("amount_coeff_of_variation").mean().item()

            mlflow.log_metric("avg_entropy_high_risk", avg_entropy_high_risk)
            mlflow.log_metric("avg_coeff_var_high_risk", avg_coeff_var_high_risk)
            print(f"Entropía promedio en alertas: {avg_entropy_high_risk:.4f}")
            print(f"Coeff. Var. promedio en alertas: {avg_coeff_var_high_risk:.4f}")
            # Esperamos que estos valores sean BAJOS para alertas de fraccionamiento

            # --- Registro de Modelos y Artefactos en MLFlow ---
            mlflow.sklearn.log_model(iso_forest, "isolation_forest_model")
            mlflow.sklearn.log_model(dbscan, "dbscan_model")
            mlflow.sklearn.log_model(scaler, "feature_scaler") # Es crucial guardar el scaler

            # Guardar el DataFrame de grupos con scores como artefacto
            # Considera guardar solo una muestra o los top N para evitar archivos muy grandes
            output_df_for_logging = candidate_groups_with_scores.sort("combined_risk_score", descending=True).head(5000) # Top 5000
            output_df_for_logging.write_parquet("high_risk_candidate_groups.parquet")
            mlflow.log_artifact("high_risk_candidate_groups.parquet")

            # Opcional: Guardar un gráfico de dispersión de los scores
            plot_data = candidate_groups_with_scores.select([
                "amount_entropy", "amount_coeff_of_variation", "combined_risk_score"
            ]).sample(n=min(50000, candidate_groups_with_scores.height)).to_pandas() # Muestra para plotear
            
            plt.figure(figsize=(10, 8))
            sns.scatterplot(x='amount_entropy', y='amount_coeff_of_variation', hue='combined_risk_score', data=plot_data, palette='viridis', alpha=0.6)
            plt.title('Ensemble Score vs. Entropía y Coeficiente de Variación')
            plt.xlabel('Entropía de los Montos')
            plt.ylabel('Coeficiente de Variación de los Montos')
            plt.colorbar(label='Combined Risk Score')
            plt.savefig("ensemble_score_scatter.png")
            mlflow.log_artifact("ensemble_score_scatter.png")
            plt.close() # Cierra la figura para no mostrarla en la ejecución

            print(f"MLFlow Run {run_id_counter} completado. View run at: {mlflow.get_tracking_uri()}")

print("\n--- Proceso de Modelado Completado ---")
print("Puedes revisar los resultados en la UI de MLFlow ejecutando: 'mlflow ui' en tu terminal.")

In [ ]:


# --- 0. Funciones Auxiliares (ya definidas o adaptadas del EDA) ---

# Función para calcular la entropía de una lista de montos
def calculate_amount_entropy(amounts: list[float]) -> float:
    if not amounts:
        return 0.0
    # Calcular value_counts y luego la entropía. np.unique es más eficiente aquí.
    unique_amounts, counts = np.unique(amounts, return_counts=True)
    probabilities = counts / counts.sum()
    return entropy(probabilities)

# Función para calcular el coeficiente de variación
def calculate_coeff_of_variation(amounts: list[float]) -> float:
    if not amounts:
        return 0.0
    amounts_arr = np.array(amounts)
    std_val = np.std(amounts_arr)
    mean_val = np.mean(amounts_arr)
    return std_val / mean_val if mean_val != 0 else 0.0

# Registrar las funciones Python como UDFs en Polars para uso con `map_batches` o `apply`
calculate_amount_entropy_pl = pl.map_batches(
    lambda s: [calculate_amount_entropy(val) for val in s],
    return_type=pl.Float64
).alias("amount_entropy")

calculate_coeff_of_variation_pl = pl.map_batches(
    lambda s: [calculate_coeff_of_variation(val) for val in s],
    return_type=pl.Float64
).alias("amount_coeff_of_variation")


# --- 1. Carga de Datos y Preprocesamiento Básico ---

# Reemplaza con la ruta real de tu archivo
FILE_PATH = "ruta/a/tu/archivo.parquet"

print("Cargando y preprocesando datos...")
try:
    df_transactions = pl.read_parquet(FILE_PATH)
    df_transactions = df_transactions.with_columns([
        pl.col("transaction_amount").cast(pl.Float64),
        pl.col("transaction_date").str.to_datetime("%Y-%m-%d %H:%M:%S%.f")
    ]).unique() # Eliminar duplicados
    print(f"Datos cargados. Dimensiones: {df_transactions.shape}")
except Exception as e:
    print(f"Error cargando o preprocesando el archivo: {e}")
    exit()

# Extraer componentes de fecha para el análisis temporal (necesario para group_by_dynamic)
df_transactions = df_transactions.with_columns([
    pl.col("transaction_date").dt.hour().alias("hour"),
    pl.col("transaction_date").dt.weekday().alias("day_of_week"),
    pl.col("transaction_date").dt.month().alias("month"),
    pl.col("transaction_date").dt.month_name().alias("month_name"),
    pl.col("transaction_date").dt.date().alias("transaction_day")
])


# --- 2. Filtrado Heurístico y Feature Engineering Avanzado ---

print("Generando grupos candidatos y features avanzadas...")

# Definir la ventana de tiempo para el fraccionamiento
WINDOW_DURATION = pl.duration(hours=24)
MIN_TRANSACTIONS_IN_GROUP = 3 # Un umbral inicial de al menos 3 transacciones

candidate_groups_df = df_transactions.group_by_dynamic(
    index_column="transaction_date",
    every=WINDOW_DURATION,
    by=["user_id", "merchant_id", "transaction_type"] # Claves para identificar el grupo
).agg([
    pl.count().alias("transaction_count"),
    pl.col("transaction_amount").sum().alias("total_group_amount"),
    pl.col("transaction_amount").std().alias("amount_stdev"),
    pl.col("transaction_amount").median().alias("amount_median"),
    pl.col("transaction_amount").min().alias("amount_min"),
    pl.col("transaction_amount").max().alias("amount_max"),
    (pl.col("transaction_date").max() - pl.col("transaction_date").min()).dt.total_minutes().alias("time_span_minutes"),
    pl.col("transaction_amount").list().alias("all_amounts_in_group"), # Para entropía/CV
    pl.col("transaction_date").list().alias("all_dates_in_group") # Para avg_time_delta
]).filter(
    pl.col("transaction_count") >= MIN_TRANSACTIONS_IN_GROUP
).with_columns([
    # Coeficiente de Variación: std / mean. Cuidado con mean=0
    (pl.col("amount_stdev") / pl.col("total_group_amount") * pl.col("transaction_count")).alias("amount_coeff_of_variation").fill_nan(0.0), # Rellenar NaN si std/mean es 0/0
    # Entropía de los Montos
    pl.col("all_amounts_in_group").apply(calculate_amount_entropy).alias("amount_entropy"),
    # Calcular promedio de delta de tiempo entre transacciones dentro del grupo
    pl.col("all_dates_in_group").apply(
        lambda dates: np.mean(np.diff(np.sort([d.timestamp() for d in dates]))) / 60 if len(dates) > 1 else 0.0
    ).alias("avg_time_delta_minutes"),
    # Flag para montos redondos (ej. múltiplos de 100000)
    pl.col("all_amounts_in_group").apply(
        lambda amounts: any(a % 100000 == 0 for a in amounts)
    ).alias("has_round_amounts")
])
# Limpiar columnas auxiliares
candidate_groups_df = candidate_groups_df.drop(["all_amounts_in_group", "all_dates_in_group"])

print(f"Total de grupos candidatos para ML: {candidate_groups_df.shape[0]}")
print(candidate_groups_df.head())


# --- 3. Preprocesamiento para Modelos ML (Normalización y Encoding) ---

print("Preparando features para los modelos ML...")

# Definir las features numéricas que serán usadas por IF y DBSCAN
# Excluir IDs o features ya agregadas que no son numéricas para el modelo
numerical_features = [
    "transaction_count",
    "total_group_amount",
    "amount_stdev",
    "amount_median",
    "amount_min",
    "amount_max",
    "time_span_minutes",
    "amount_coeff_of_variation",
    "amount_entropy",
    "avg_time_delta_minutes"
]

# Asegurarse de que todas las columnas existan antes de seleccionar
numerical_features_exist = [col for col in numerical_features if col in candidate_groups_df.columns]
if len(numerical_features_exist) != len(numerical_features):
    print(f"Advertencia: Algunas features numéricas esperadas no existen. Usando: {numerical_features_exist}")
numerical_features = numerical_features_exist

X_candidates_polars = candidate_groups_df.select(numerical_features)

# Convertir a NumPy para Scikit-learn
X_candidates = X_candidates_polars.to_numpy()

# Normalización con StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_candidates)
print(f"Features escaladas. Dimensiones de X_scaled: {X_scaled.shape}")


# --- 4. Ensamble de Modelos (Isolation Forest + DBSCAN) con MLFlow ---

# Configuración de MLFlow
# Si usas un servidor MLFlow remoto, configura mlflow.set_tracking_uri()
mlflow.set_experiment("Deteccion_Fraccionamiento_Nequi")

# Parámetros para iterar (ejemplo, puedes añadir más o usar un grid search más formal)
if_params_list = [
    {"n_estimators": 100, "contamination": 'auto', "random_state": 42},
    {"n_estimators": 200, "contamination": 0.01, "random_state": 42},
]

dbscan_params_list = [
    {"eps": 0.5, "min_samples": 5},
    {"eps": 0.3, "min_samples": 10},
]

# Iterar sobre las combinaciones de parámetros para encontrar el mejor modelo
run_id_counter = 0
for if_params in if_params_list:
    for dbscan_params in dbscan_params_list:
        run_id_counter += 1
        with mlflow.start_run(run_name=f"Run_{run_id_counter}"):
            print(f"\n--- Ejecutando MLFlow Run {run_id_counter} ---")
            print(f"IF Params: {if_params}")
            print(f"DBSCAN Params: {dbscan_params}")

            # Logear parámetros en MLFlow
            mlflow.log_params({"if_" + k: v for k, v in if_params.items()})
            mlflow.log_params({"dbscan_" + k: v for k, v in dbscan_params.items()})

            # --- Entrenamiento de Isolation Forest ---
            iso_forest = IsolationForest(**if_params)
            iso_forest.fit(X_scaled)
            # anomaly_scores_if: Menor valor = Más anómalo
            anomaly_scores_if = iso_forest.decision_function(X_scaled)
            # Convertir a un score de riesgo: Mayor valor = Más riesgo (normalizar 0-1)
            min_score, max_score = anomaly_scores_if.min(), anomaly_scores_if.max()
            risk_scores_if = 1 - (anomaly_scores_if - min_score) / (max_score - min_score)
            
            # --- Entrenamiento de DBSCAN ---
            dbscan = DBSCAN(**dbscan_params)
            # labels: -1 para ruido (anomalías), 0, 1, ... para clústeres
            dbscan_labels = dbscan.fit_predict(X_scaled)
            # is_noise: 1 si es ruido, 0 si es parte de un clúster
            is_dbscan_noise = (dbscan_labels == -1).astype(int)

            # --- Ensamble de Scores ---
            # Score combinado: Puedes ponderar o usar una regla lógica (ej. AND)
            # Aquí usaremos una combinación simple de los scores normalizados
            # Ponderación 0.7 para IF, 0.3 para DBSCAN (ajustable)
            combined_score = 0.7 * risk_scores_if + 0.3 * is_dbscan_noise

            # Añadir scores al DataFrame de candidatos (para análisis posterior)
            candidate_groups_with_scores = candidate_groups_df.with_columns([
                pl.Series("isolation_forest_score", anomaly_scores_if),
                pl.Series("dbscan_label", dbscan_labels),
                pl.Series("is_dbscan_noise", is_dbscan_noise),
                pl.Series("combined_risk_score", combined_score)
            ])

            # --- Evaluación y Registro de Métricas ---
            # Para modelos no supervisados sin ground truth, la evaluación es desafiante.
            # Aquí usaremos métricas intrínsecas y un umbral para simular Precision/Recall.

            # Métrica intrínseca de clustering (para DBSCAN)
            # silhouette_score requiere al menos 2 clústeres y 2 muestras.
            try:
                if len(np.unique(dbscan_labels)) > 1 and len(dbscan_labels) > 1:
                    silhouette_avg = silhouette_score(X_scaled, dbscan_labels)
                    mlflow.log_metric("dbscan_silhouette_score", silhouette_avg)
                    print(f"DBSCAN Silhouette Score: {silhouette_avg:.4f}")
                else:
                    print("No suficientes clústeres para calcular Silhouette Score.")
            except Exception as e:
                print(f"Error calculando Silhouette Score: {e}")

            # Umbral de riesgo combinado (para simular alertas)
            # Por ejemplo, top 5% de los scores más altos son alertas
            alert_threshold = np.percentile(combined_score, 95) # Top 5% más riesgoso
            num_alerts = (combined_score >= alert_threshold).sum()
            
            mlflow.log_metric("alert_threshold", alert_threshold)
            mlflow.log_metric("num_alerts", num_alerts)
            print(f"Alertas generadas (top 5%): {num_alerts}")

            # Evaluación de la "Calidad" de las Alertas sin etiquetas reales:
            # Puedes examinar las características de los grupos de "alto riesgo"
            high_risk_groups = candidate_groups_with_scores.filter(
                pl.col("combined_risk_score") >= alert_threshold
            )
            
            # Puedes calcular la "uniformidad promedio" de los montos en los grupos de alto riesgo
            avg_entropy_high_risk = high_risk_groups.select("amount_entropy").mean().item()
            avg_coeff_var_high_risk = high_risk_groups.select("amount_coeff_of_variation").mean().item()

            mlflow.log_metric("avg_entropy_high_risk", avg_entropy_high_risk)
            mlflow.log_metric("avg_coeff_var_high_risk", avg_coeff_var_high_risk)
            print(f"Entropía promedio en alertas: {avg_entropy_high_risk:.4f}")
            print(f"Coeff. Var. promedio en alertas: {avg_coeff_var_high_risk:.4f}")
            # Esperamos que estos valores sean BAJOS para alertas de fraccionamiento

            # --- Registro de Modelos y Artefactos en MLFlow ---
            mlflow.sklearn.log_model(iso_forest, "isolation_forest_model")
            mlflow.sklearn.log_model(dbscan, "dbscan_model")
            mlflow.sklearn.log_model(scaler, "feature_scaler") # Es crucial guardar el scaler

            # Guardar el DataFrame de grupos con scores como artefacto
            # Considera guardar solo una muestra o los top N para evitar archivos muy grandes
            output_df_for_logging = candidate_groups_with_scores.sort("combined_risk_score", descending=True).head(5000) # Top 5000
            output_df_for_logging.write_parquet("high_risk_candidate_groups.parquet")
            mlflow.log_artifact("high_risk_candidate_groups.parquet")

            # Opcional: Guardar un gráfico de dispersión de los scores
            plot_data = candidate_groups_with_scores.select([
                "amount_entropy", "amount_coeff_of_variation", "combined_risk_score"
            ]).sample(n=min(50000, candidate_groups_with_scores.height)).to_pandas() # Muestra para plotear
            
            plt.figure(figsize=(10, 8))
            sns.scatterplot(x='amount_entropy', y='amount_coeff_of_variation', hue='combined_risk_score', data=plot_data, palette='viridis', alpha=0.6)
            plt.title('Ensemble Score vs. Entropía y Coeficiente de Variación')
            plt.xlabel('Entropía de los Montos')
            plt.ylabel('Coeficiente de Variación de los Montos')
            plt.colorbar(label='Combined Risk Score')
            plt.savefig("ensemble_score_scatter.png")
            mlflow.log_artifact("ensemble_score_scatter.png")
            plt.close() # Cierra la figura para no mostrarla en la ejecución

            print(f"MLFlow Run {run_id_counter} completado. View run at: {mlflow.get_tracking_uri()}")

print("\n--- Proceso de Modelado Completado ---")
print("Puedes revisar los resultados en la UI de MLFlow ejecutando: 'mlflow ui' en tu terminal.")